# Chapter 21 — The Agent Cannot Grade Its Own Homework

**Companion to *Applied AI*.**

A check ran and passed. Does that verify the claim? This notebook shows why
that question has three axes instead of one — with real checks that pass,
fail, and refuse to run, and a real checker that passed a wrong answer while
rejecting right ones.

## Question

**Independence, adequacy, binding — which one failed?**

## What this notebook does

It **inspects** the pinned ten-check run
(`verification-binding/2026-09-14-1b3c7a2/`, PASS / FAIL / ERROR side by
side) and **reproduces** the chapter's adequacy cases from the Stage 29B
ladder bundle (`execution-ladder/2026-09-14-7a0d43b/`): item A05, accepted
and wrong; items T07 and T10, correct and rejected.

```text
check passed  ≠  claim verified
```

## Setup

Standard library only. No network, no API key, no `codeai` import.
Only bundle-relative paths are shown; override the evidence root with
`APPLIED_AI_EVIDENCE`.

In [1]:
import json
import os
from pathlib import Path

def find_evidence_dir(marker="verification-binding"):
    """Locate the preserved evidence. Override with APPLIED_AI_EVIDENCE."""
    env = os.environ.get("APPLIED_AI_EVIDENCE")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "evidence",
                     base / "experiments" / "applied-ai" / "evidence"):
            if (cand / marker).is_dir():
                return cand
    raise FileNotFoundError(
        "Preserved evidence not found. Set APPLIED_AI_EVIDENCE to the "
        "directory holding the Applied AI evidence bundles.")

EVIDENCE_DIR = find_evidence_dir()
VB = EVIDENCE_DIR / "verification-binding" / "2026-09-14-1b3c7a2"
LAD = EVIDENCE_DIR / "execution-ladder" / "2026-09-14-7a0d43b"
print("bundles: verification-binding/2026-09-14-1b3c7a2")
print("         execution-ladder/2026-09-14-7a0d43b")
checks = {c["check_id"]: c for c in
          json.loads((VB / "results.json").read_text(encoding="utf-8"))}
lad = json.loads((LAD / "analysis.json").read_text(encoding="utf-8"))
rows = {r["item"]: r for r in lad["arms"]["ladder"]["rows"]}
print("checks:", len(checks), "| ladder items:", len(rows))

bundles: verification-binding/2026-09-14-1b3c7a2
         execution-ladder/2026-09-14-7a0d43b
checks: 10 | ladder items: 40


## 1. Independence: a second observer that can say no

The generator's success claim is not verification. The negative-control pair
runs one checker twice: correct bytes pass, a seeded defect fails. The FAIL
verdict with preserved stdout is what independence buys — not correctness,
just a separate check that refuses to take the first observer's word.

In [2]:
for cid in ("chk-neg-pass", "chk-neg-fail"):
    c = checks[cid]
    print(f"{cid:<14}verdict={c['verdict']:<6}exit={c['exit_code']}  invocations={c['invocations']}")
assert checks["chk-neg-pass"]["verdict"] == "PASS"
assert checks["chk-neg-fail"]["verdict"] == "FAIL"
print()
print("Same checker, opposite verdicts. Separation alone says nothing about")
print("whether the check is adequate — that is the next axis.")

chk-neg-pass  verdict=PASS  exit=0  invocations=1
chk-neg-fail  verdict=FAIL  exit=1  invocations=1

Same checker, opposite verdicts. Separation alone says nothing about
whether the check is adequate — that is the next axis.


## 2. Adequacy, direction one: a PASS that should have been a FAIL

Two oracles name the same claim `c-ttl-21`. The weak oracle checks presence
and passes — appending REPRODUCED evidence. The strong oracle checks exact
bytes and fails — appending a REFUTED status. The claim effects are
directional and ordered in the ledger: PASS adds evidence, FAIL adds
refutation, ERROR adds neither.

In [3]:
for cid in ("chk-weak", "chk-strong"):
    c = checks[cid]
    print(f"{cid:<12}verdict={c['verdict']:<6}claims={c['claim_ids']}")
assert checks["chk-weak"]["verdict"] == "PASS"
assert checks["chk-strong"]["verdict"] == "FAIL"
assert checks["chk-weak"]["claim_ids"] == checks["chk-strong"]["claim_ids"] == ["c-ttl-21"]
print()
print("One claim, two verdicts. 'A check passed' cannot mean 'the claim is")
print("verified' when the check tested presence and the property needed bytes.")

chk-weak    verdict=PASS  claims=['c-ttl-21']
chk-strong  verdict=FAIL  claims=['c-ttl-21']

One claim, two verdicts. 'A check passed' cannot mean 'the claim is
verified' when the check tested presence and the property needed bytes.


## 3. Adequacy, direction two: the grounded answer that was wrong

Stage 29B's TTL checker `ttl-check-v1` is deliberately narrow and frozen:
the output must parse as one JSON object with an integer `seconds` and a
non-empty `quote`; the quote must appear verbatim in the input, hold exactly
one duration, and the seconds must follow from it.

Item A05 states **two** TTLs — 1 minute for staging, 10 minutes for
production — with no single correct answer. The free rung answered
`{"seconds": 600, "quote": "10 minutes"}`. The quote is verbatim, holds one
duration, and 600 follows from it. The check passed. The answer was accepted.
And wrong.

In [4]:
a05 = rows["A05"]
print("item      : A05 (category: %s)" % a05["category"])
print("answer    : 600 seconds from quote '10 minutes' -> check PASSED")
print("accepted  :", a05["accepted"], "| rung:", a05["resolved_rung"])
print("gold      : no single value (input ambiguous)")
print("negative_acceptance (accepted AND wrong):", a05["negative_acceptance"])
assert a05["accepted"] and a05["negative_acceptance"]
assert a05["resolved_rung"] == "zen-free" and a05["gold_seconds"] is None
print()
print("Grounding and derivability were verified; ambiguity and relevance were")
print("not. The check answered 'does this value follow from a quoted passage?'")
print("Acceptance needed 'does this input determine one value?' — a different")
print("property, and no parser tuning closes the gap without changing what the")
print("check claims to test.")

item      : A05 (category: no-single-ttl)
answer    : 600 seconds from quote '10 minutes' -> check PASSED
accepted  : True | rung: zen-free
gold      : no single value (input ambiguous)
negative_acceptance (accepted AND wrong): True

Grounding and derivability were verified; ambiguity and relevance were
not. The check answered 'does this value follow from a quoted passage?'
Acceptance needed 'does this input determine one value?' — a different
property, and no parser tuning closes the gap without changing what the
check claims to test.


## 4. Adequacy, direction three: correct answers, rejected

T07 reads `Cache TTL: 7200 (seconds)` with gold 7200 — the parser finds no
duration expression in that shape and declines every correct answer as
`quote_has_0_durations`. T10 reads `1 hour 30 minutes` with gold 5400 — two
durations, declined as `quote_has_2_durations`. Both went to a person after
repeated model calls. Correct answer ≠ answer verifiable by this checker.

In [5]:
for item in ("T07", "T10"):
    r = rows[item]
    print(f"{item}: gold={r['gold_seconds']:<6} accepted={r['accepted']!s:<6}",
          f"model_calls={r['model_calls']} status={r['status']}")
    print(f"     decline reasons: {sorted(set(r['decline_reasons']))}")
assert not rows["T07"]["accepted"] and rows["T07"]["asked_person"]
assert not rows["T10"]["accepted"] and rows["T10"]["asked_person"]
assert "check_failed:quote_has_0_durations" in rows["T07"]["decline_reasons"]
assert "check_failed:quote_has_2_durations" in rows["T10"]["decline_reasons"]
print()
print("A05 was independent and bound but inadequate; T07 and T10 were")
print("independent and bound with an inadequate parser in the other direction.")
print("None of this shortens to 'the verifier worked' or 'the verifier failed'.")

T07: gold=7200   accepted=False  model_calls=3 status=asked_human
     decline reasons: ['check_failed:quote_has_0_durations', 'rule_declined:0 ttl lines']
T10: gold=5400   accepted=False  model_calls=3 status=asked_human
     decline reasons: ['check_failed:quote_has_2_durations', 'rule_declined:0 ttl lines']

A05 was independent and bound but inadequate; T07 and T10 were
independent and bound with an inadequate parser in the other direction.
None of this shortens to 'the verifier worked' or 'the verifier failed'.


## 5. Binding: refusing to check the wrong state

Requested state `state-B`, observed `state-A`: the verifier never runs.
Unavailable resolver: no observation is invented, no verifier call. These are
ERRORs — and a broken checker is not a failed artifact. A verifier that
crashes, times out, or is missing yields ERROR, never PASS and never FAIL.

In [6]:
print(f"{'check':<18}{'verdict':<8}{'invocations':<12}note")
print("-" * 70)
notes = {"chk-stale": "requested state-B, observed state-A",
         "chk-unavail-none": "no resolver configured",
         "chk-unavail-raise": "resolver raised RuntimeError",
         "chk-verifier-exc": "verifier raised boom",
         "chk-timeout": "30s sleep, 1s budget",
         "chk-nobinary": "binary missing, no exit code"}
for cid, note in notes.items():
    c = checks[cid]
    print(f"{cid:<18}{c['verdict']:<8}{c['invocations']:<12}{note}")
    assert c["verdict"] == "ERROR", cid
assert checks["chk-stale"]["invocations"] == 0
print()
print("chk-stale error:", checks["chk-stale"]["error"][:90], "...")
print("One reading, refused before invocation, observation preserved on the")
print("record. An attempt is not standing (Chapter 18): INCONCLUSIVE or ERROR")
print("moves no claim.")

check             verdict invocations note
----------------------------------------------------------------------
chk-stale         ERROR   0           requested state-B, observed state-A
chk-unavail-none  ERROR   0           no resolver configured
chk-unavail-raise ERROR   0           resolver raised RuntimeError
chk-verifier-exc  ERROR   1           verifier raised boom
chk-timeout       ERROR   1           30s sleep, 1s budget
chk-nobinary      ERROR   1           binary missing, no exit code

chk-stale error: target state mismatch: expected 2b21b39d573e5ff45aeb52f161bab2db821b4ea72d13d4eb0d7c9a5f8b ...
One reading, refused before invocation, observation preserved on the
record. An attempt is not standing (Chapter 18): INCONCLUSIVE or ERROR
moves no claim.


## Interpretation

1. **Independence.** The generator's claim is not verification. A separate
   check can contradict it — separation buys a second observer, not truth.
2. **Adequacy.** A check can be separate, correctly bound, and still test the
   wrong property: grounding instead of ambiguity (A05), strictness where the
   property needed leniency (T07, T10). Nothing on a request certifies
   adequacy; a seeded defect shows sensitivity to one failure class, not
   adequacy to the task.
3. **Binding.** When state binding is requested: one reading, refuse
   unavailable or mismatched state before invoking the verifier, preserve the
   reading used. Neither binding freezes concurrent state.
4. **ERROR ≠ FAIL.** A broken checker is not a failed artifact. A check that
   could not run moves no claim — letting it revoke a decision would hand an
   infrastructure failure the power to withdraw a justification.
5. **Preserved limits.** The shipped verifier returns only PASS/FAIL by exit
   code and ERROR otherwise — the stage records the INCONCLUSIVE gap instead
   of fabricating a case. Adequacy is still not enforced.

## Try it yourself

1. In `results.json`, compare `observed` for `chk-stale` with its error
   string: which hash is the runtime's reading, and which was requested?
2. Write the check A05 needed ("does this input determine one value?") in
   three lines of pseudocode. What does it need that `ttl-check-v1` never had?
3. Loosen the parser so T07's `7200 (seconds)` verifies. Does A05 still fail
   safe? Adequacy trades off — show it.

*Evidence: `experiments/applied-ai/evidence/verification-binding/2026-09-14-1b3c7a2/`
(ten checks, stdlib-only verifier, 5/5 seeded corruptions rejected) and
`experiments/applied-ai/evidence/execution-ladder/2026-09-14-7a0d43b/`
(`analysis.json` ladder rows). No network, no API key, no `codeai` import.*